# Filter Dataset & Column Review

**Goal:** Create a clean, filtered parquet for the thesis model and review every column group so we can decide what stays and what goes in the next step.

**Filters applied:**
1. **Within-league transfers only** → `from_competition == to_competition`
2. **Actual team change** → `from_team_id != to_team_id`
3. **Same position pre/post** → `from_position == to_position`
4. **Minimum 900 minutes** on both sides → `from_Minutes >= 900` and `to_Minutes >= 900`

**This notebook does NOT drop columns** — it only saves the filtered rows and presents each column group for human review.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_PATH = "../../../thesis_data/processed_data/master_dataset/transfers_model_2018_2025.parquet"
OUTPUT_DIR = "../../../thesis_data/processed_data/thesis_model_dataset"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "within_league_transfers.parquet")

df = pd.read_parquet(DATA_PATH)
print(f"Raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

Raw dataset: 262,340 rows x 539 columns


## 1. Apply Filters (step by step)

In [2]:
n0 = len(df)

# Filter 1: within-league (same competition)
mask_same_comp = df["from_competition"] == df["to_competition"]
n1 = mask_same_comp.sum()

# Filter 2: actual team change
mask_diff_team = df["from_team_id"] != df["to_team_id"]
n2 = (mask_same_comp & mask_diff_team).sum()

# Filter 3: same position pre and post
mask_same_pos = df["from_position"] == df["to_position"]
n3 = (mask_same_comp & mask_diff_team & mask_same_pos).sum()

# Filter 4: minimum 900 minutes on both sides
mask_min_minutes = (df["from_Minutes"] >= 900) & (df["to_Minutes"] >= 900)
mask_all = mask_same_comp & mask_diff_team & mask_same_pos & mask_min_minutes
n4 = mask_all.sum()

print("Filter funnel:")
print(f"  Raw data:                        {n0:>8,}")
print(f"  + Within-league only:            {n1:>8,}  (dropped {n0 - n1:,})")
print(f"  + Different team:                {n2:>8,}  (dropped {n1 - n2:,})")
print(f"  + Same position from/to:         {n3:>8,}  (dropped {n2 - n3:,})")
print(f"  + Min 900 min both sides:        {n4:>8,}  (dropped {n3 - n4:,})")
print(f"\n  Final: {n4:,} rows  ({100*n4/n0:.1f}% of original)")

Filter funnel:
  Raw data:                         262,340
  + Within-league only:             184,149  (dropped 78,191)
  + Different team:                  48,043  (dropped 136,106)
  + Same position from/to:           36,915  (dropped 11,128)
  + Min 900 min both sides:          20,043  (dropped 16,872)

  Final: 20,043 rows  (7.6% of original)


In [3]:
fdf = df.loc[mask_all].copy().reset_index(drop=True)
print(f"Filtered dataset: {fdf.shape[0]:,} rows x {fdf.shape[1]} columns")
print(f"Unique players: {fdf['wy_player_id'].nunique():,}")
print(f"Seasons: {sorted(fdf['from_season'].unique())}")
print(f"Positions: {sorted(fdf['from_position'].unique())}")
print(f"Competitions: {fdf['from_comp_name'].nunique()} unique")

Filtered dataset: 20,043 rows x 539 columns
Unique players: 16,102
Seasons: [np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024)]
Positions: ['Central Defender', 'Full Back', 'Goalkeeper', 'Midfielder', 'Striker', 'Winger']
Competitions: 134 unique


## 2. Save Filtered Parquet

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
fdf.to_parquet(OUTPUT_PATH, index=False)
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved to: {OUTPUT_PATH}")
print(f"File size: {size_mb:.1f} MB")

Saved to: ../../../thesis_data/processed_data/thesis_model_dataset/within_league_transfers.parquet
File size: 44.1 MB


---

## 3. Column-by-Column Review for Modeling

Below, each column group is listed with its columns, basic stats, and a **recommendation** tag:
- `DROP` — clearly not useful for modeling (IDs, URLs, redundant after filtering)
- `KEEP` — likely useful
- `REVIEW` — needs human decision (could go either way)

**Nothing is dropped here.** This is a menu for you to decide.

### 3.1 Player Identity Columns

In [5]:
identity_cols = [
    "wy_player_id", "wyscout_birth_country", "wyscout_first_name", "wyscout_foot",
    "wyscout_height", "wyscout_image_url", "wyscout_last_name", "wyscout_passport",
    "wyscout_role", "wyscout_weight", "short_name", "birth_date", "player_season_age",
]

review = []
for col in identity_cols:
    nuniq = fdf[col].nunique()
    nulls = fdf[col].isnull().sum()
    dtype = str(fdf[col].dtype)

    # Tagging logic
    if col == "wyscout_image_url":
        tag = "DROP"
        reason = "URL, no modeling value"
    elif col in ("wyscout_first_name", "wyscout_last_name", "short_name"):
        tag = "DROP"
        reason = "Name string, only useful as ID/label"
    elif col == "wy_player_id":
        tag = "DROP"
        reason = "Internal ID — keep only as index/join key, not a feature"
    elif col == "birth_date":
        tag = "DROP"
        reason = "Redundant with player_season_age"
    elif col == "wyscout_passport":
        tag = "REVIEW"
        reason = "Nationality info — could encode or drop"
    elif col == "wyscout_birth_country":
        tag = "REVIEW"
        reason = "Birth country — could encode or drop"
    elif col == "wyscout_role":
        tag = "REVIEW"
        reason = "High-level role — redundant with from_position?"
    elif col in ("wyscout_height", "wyscout_weight"):
        tag = "KEEP"
        reason = "Physical attribute, potentially informative"
    elif col == "wyscout_foot":
        tag = "REVIEW"
        reason = "Preferred foot — could encode"
    elif col == "player_season_age":
        tag = "KEEP"
        reason = "Age is a key predictor"
    else:
        tag = "REVIEW"
        reason = ""

    review.append({"column": col, "dtype": dtype, "nunique": nuniq, "nulls": nulls, "tag": tag, "reason": reason})

pd.DataFrame(review).set_index("column")

,dtype,nunique,nulls,tag,reason
column,,,,,
wy_player_id,int32,16102,0,DROP,"Internal ID — keep only as index/join key, not..."
wyscout_birth_country,object,116,15313,REVIEW,Birth country — could encode or drop
wyscout_first_name,object,2308,15313,DROP,"Name string, only useful as ID/label"
wyscout_foot,object,3,15389,REVIEW,Preferred foot — could encode
wyscout_height,float64,45,15313,KEEP,"Physical attribute, potentially informative"
wyscout_image_url,object,3534,15313,DROP,"URL, no modeling value"
wyscout_last_name,object,3457,15313,DROP,"Name string, only useful as ID/label"
wyscout_passport,object,137,15313,REVIEW,Nationality info — could encode or drop
wyscout_role,object,4,15313,REVIEW,High-level role — redundant with from_position?


### 3.2 Transfer Metadata Columns

In [6]:
transfer_cols = [
    "transfer_type", "competition_start_date", "first_played_date", "last_played_date",
    "tm_player_id", "tm_transfer_date", "tm_transfer_value", "tm_transfer_fee",
    "tm_remaining_contract_days", "tm_contract_until_date",
]

review = []
for col in transfer_cols:
    nuniq = fdf[col].nunique()
    nulls = fdf[col].isnull().sum()
    dtype = str(fdf[col].dtype)

    if col == "transfer_type":
        tag = "DROP"
        reason = "Constant after filtering (all are same_competition)"
    elif col == "tm_player_id":
        tag = "DROP"
        reason = "Transfermarkt ID — join key, not a feature"
    elif col in ("competition_start_date", "first_played_date", "last_played_date",
                 "tm_transfer_date", "tm_contract_until_date"):
        tag = "DROP"
        reason = "Raw date — not directly usable as feature (info captured by season/age/contract_days)"
    elif col == "tm_transfer_value":
        tag = "REVIEW"
        reason = "Market value at transfer — could be target or feature"
    elif col == "tm_transfer_fee":
        tag = "REVIEW"
        reason = "Actual fee paid — could be target or feature"
    elif col == "tm_remaining_contract_days":
        tag = "KEEP"
        reason = "Contract situation affects transfer dynamics"
    else:
        tag = "REVIEW"
        reason = ""

    review.append({"column": col, "dtype": dtype, "nunique": nuniq, "nulls": nulls,
                    "null_pct": f"{100*nulls/len(fdf):.1f}%", "tag": tag, "reason": reason})

pd.DataFrame(review).set_index("column")

,dtype,nunique,nulls,null_pct,tag,reason
column,,,,,,
transfer_type,object,1,0,0.0%,DROP,Constant after filtering (all are same_competi...
competition_start_date,object,467,0,0.0%,DROP,Raw date — not directly usable as feature (inf...
first_played_date,datetime64[ns],6755,0,0.0%,DROP,Raw date — not directly usable as feature (inf...
last_played_date,datetime64[ns],6952,0,0.0%,DROP,Raw date — not directly usable as feature (inf...
tm_player_id,float64,14662,1696,8.5%,DROP,"Transfermarkt ID — join key, not a feature"
tm_transfer_date,datetime64[us],1598,2165,10.8%,DROP,Raw date — not directly usable as feature (inf...
tm_transfer_value,Float64,126,2165,10.8%,REVIEW,Market value at transfer — could be target or ...
tm_transfer_fee,Float64,481,7208,36.0%,REVIEW,Actual fee paid — could be target or feature
tm_remaining_contract_days,float64,550,16498,82.3%,KEEP,Contract situation affects transfer dynamics


### 3.3 Structural / Context Columns (from\_ & to\_ non-metric)

These are the "envelope" columns for each side: team ID, competition ID, season, position, minutes.

In [7]:
structural_cols = [
    "from_Minutes", "from_competition", "from_position", "from_season", "from_team_id",
    "to_Minutes", "to_competition", "to_position", "to_season", "to_team_id",
]

review = []
for col in structural_cols:
    nuniq = fdf[col].nunique()
    nulls = fdf[col].isnull().sum()
    dtype = str(fdf[col].dtype)

    if col in ("from_competition", "to_competition"):
        tag = "DROP"
        reason = "Same value for both (within-league filter) — redundant"
    elif col == "to_position":
        tag = "DROP"
        reason = "Same as from_position (filter applied)"
    elif col in ("from_team_id", "to_team_id"):
        tag = "DROP"
        reason = "Team ID — join key, not a direct feature (team stats capture this)"
    elif col in ("from_Minutes", "to_Minutes"):
        tag = "REVIEW"
        reason = "Playing time — feature or just a filter? Could normalize metrics"
    elif col == "from_position":
        tag = "KEEP"
        reason = "Position is a key grouping variable"
    elif col in ("from_season", "to_season"):
        tag = "REVIEW"
        reason = "Season — temporal info, could be useful or cause leakage"
    else:
        tag = "REVIEW"
        reason = ""

    review.append({"column": col, "dtype": dtype, "nunique": nuniq, "nulls": nulls, "tag": tag, "reason": reason})

pd.DataFrame(review).set_index("column")

,dtype,nunique,nulls,tag,reason
column,,,,,
from_Minutes,int32,2539,0,REVIEW,Playing time — feature or just a filter? Could...
from_competition,int32,247,0,DROP,Same value for both (within-league filter) — r...
from_position,object,6,0,KEEP,Position is a key grouping variable
from_season,int16,7,0,REVIEW,"Season — temporal info, could be useful or cau..."
from_team_id,int32,3817,0,DROP,"Team ID — join key, not a direct feature (team..."
to_Minutes,int32,2530,0,REVIEW,Playing time — feature or just a filter? Could...
to_competition,int32,247,0,DROP,Same value for both (within-league filter) — r...
to_position,object,6,0,DROP,Same as from_position (filter applied)
to_season,int16,7,0,REVIEW,"Season — temporal info, could be useful or cau..."


### 3.4 Competition Metadata (from\_comp\_ & to\_comp\_)

Information about the league/season context on each side.

In [8]:
comp_cols_from = [c for c in fdf.columns if c.startswith("from_comp_")]
comp_cols_to = [c for c in fdf.columns if c.startswith("to_comp_")]

review = []
for col in comp_cols_from + comp_cols_to:
    nuniq = fdf[col].nunique()
    nulls = fdf[col].isnull().sum()
    dtype = str(fdf[col].dtype)
    prefix = "from" if col.startswith("from_") else "to"
    base = col.replace("from_comp_", "").replace("to_comp_", "")

    if base in ("start_date", "end_date"):
        tag = "DROP"
        reason = "Raw date — season name/ID captures this"
    elif base == "completed":
        tag = "DROP"
        reason = "Boolean for season completion — likely always True after filtering"
    elif base == "season_id":
        tag = "DROP"
        reason = "Internal ID — redundant with season_name"
    elif base == "season_name":
        tag = "REVIEW"
        reason = "Season label — redundant with from_season?"
    elif base == "name":
        tag = "REVIEW"
        reason = f"Competition name — same for both sides (within-league). Could encode as feature for league level"
    elif base == "country":
        tag = "REVIEW"
        reason = "Country of competition — same both sides. Could encode"
    elif base == "division":
        tag = "REVIEW"
        reason = "Division level (1st, 2nd) — potentially useful as league quality proxy"
    else:
        tag = "REVIEW"
        reason = ""

    review.append({"column": col, "dtype": dtype, "nunique": nuniq, "nulls": nulls, "tag": tag, "reason": reason})

pd.DataFrame(review).set_index("column")

,dtype,nunique,nulls,tag,reason
column,,,,,
from_comp_completed,object,1,5939,DROP,Boolean for season completion — likely always ...
from_comp_country,object,97,5939,REVIEW,Country of competition — same both sides. Coul...
from_comp_division,float64,3,5939,REVIEW,"Division level (1st, 2nd) — potentially useful..."
from_comp_end_date,datetime64[ns],429,5939,DROP,Raw date — season name/ID captures this
from_comp_name,object,134,5939,REVIEW,Competition name — same for both sides (within...
from_comp_season_id,float64,887,5939,DROP,Internal ID — redundant with season_name
from_comp_season_name,object,15,5939,REVIEW,Season label — redundant with from_season?
from_comp_start_date,datetime64[ns],419,5939,DROP,Raw date — season name/ID captures this
to_comp_completed,object,2,5939,DROP,Boolean for season completion — likely always ...


### 3.5 Performance Metrics — Quality Scores (Wyscout composites)

These are Wyscout's pre-computed quality scores (e.g., "Active defence", "Box threat"). They exist for both `from_` and `to_` sides.

**Key question:** Do we use these composite scores, the raw per-90 metrics, or the z-scores? Using all three is redundant.

In [9]:
# Identify the three metric layers for "from_" side
all_from_perf = [c for c in fdf.columns if c.startswith("from_")
                 and not c.startswith("from_z_score_")
                 and not c.startswith("from_team_stats_")
                 and not c.startswith("from_comp_")
                 and c not in ["from_Minutes", "from_competition", "from_position",
                               "from_season", "from_team_id"]]

quality_scores = [c for c in all_from_perf if "per 90" not in c and "pct" not in c.lower()
                  and c not in ["from_matches_pct", "from_minutes_pct"]]
per90_metrics = [c for c in all_from_perf if "per 90" in c]
pct_metrics = [c for c in all_from_perf if c in ["from_matches_pct", "from_minutes_pct"]
               or ("%" in c and c not in quality_scores)]
other = [c for c in all_from_perf if c not in quality_scores and c not in per90_metrics and c not in pct_metrics]

print(f"Quality/composite scores: {len(quality_scores)}")
print(f"Per-90 counting metrics:  {len(per90_metrics)}")
print(f"Percentage metrics:       {len(pct_metrics)}")
print(f"Other:                    {len(other)}")
if other:
    print(f"  → {other}")
print()
print("--- Quality scores (from_) ---")
for c in sorted(quality_scores):
    print(f"  {c.replace('from_', '')}")

Quality/composite scores: 48
Per-90 counting metrics:  46
Percentage metrics:       2
Other:                    0

--- Quality scores (from_) ---
  Active defence
  Aerial threat
  Aerials won %
  Attacking aerials won %
  Box threat
  Carries (xT) per 100 receptions
  Chance prevention
  Composure
  Defending 1v1 %
  Defensive aerials won %
  Defensive area (m^2)
  Defensive duels won %
  Defensive heading
  Defensive line height (m)
  Dribbles success %
  Dribbling
  Effectiveness
  Finishing
  Goals - xG
  Goals per box touch
  High turnovers per low reception
  Hold-up play
  Intelligent defence
  Involvement
  Opposition pass success % into defensive area
  Opposition progressive passes from defensive area %
  Opposition xG after defensive action
  Opposition xG from defensive area
  Opposition xT from defensive area
  Opposition xT into defensive area
  Passes (xT) per 100 receptions
  Passing quality
  Playing time
  Poaching
  Possessions won per opponent possession
  Pressing


In [10]:
# Summary stats for quality scores — check for constants, near-zero variance, or heavy nulls
qs_stats = fdf[quality_scores].describe().T[["count", "mean", "std", "min", "max"]]
qs_stats["null_pct"] = (1 - qs_stats["count"] / len(fdf)) * 100
qs_stats["cv"] = (qs_stats["std"] / qs_stats["mean"].abs()).replace([np.inf, -np.inf], np.nan)
qs_stats = qs_stats.rename(index=lambda x: x.replace("from_", ""))

print("Quality scores summary (from_ side):")
print("  Tag: REVIEW — decide whether to keep composites, per-90s, z-scores, or a combination")
print()
qs_stats[["null_pct", "mean", "std", "min", "max"]]

Quality scores summary (from_ side):
  Tag: REVIEW — decide whether to keep composites, per-90s, z-scores, or a combination



,null_pct,mean,std,min,max
Active defence,4.111,0.050,0.675,-2.394,3.532
Aerial threat,9.679,0.069,0.668,-2.333,3.564
Aerials won %,0.020,0.516,0.180,0.000,1.000
Attacking aerials won %,9.215,0.567,0.197,0.000,1.000
Box threat,9.609,0.011,0.664,-1.635,4.059
Carries (xT) per 100 receptions,0.000,0.246,0.117,0.012,1.640
Chance prevention,57.202,-0.156,0.587,-2.994,1.933
Composure,0.005,0.026,0.560,-2.273,3.058
Defending 1v1 %,3.742,0.479,0.199,0.000,1.000
Defensive aerials won %,0.110,0.486,0.218,0.000,1.000


### 3.6 Performance Metrics — Per-90 Counting Stats

In [11]:
p90_stats = fdf[per90_metrics].describe().T[["count", "mean", "std", "min", "max"]]
p90_stats["null_pct"] = (1 - p90_stats["count"] / len(fdf)) * 100
p90_stats = p90_stats.rename(index=lambda x: x.replace("from_", ""))

print(f"Per-90 metrics: {len(per90_metrics)} columns")
print("  Tag: REVIEW — these are the raw counting stats normalized to 90 min.")
print("  If z-scores are kept, these may be redundant (z-scores = position-adjusted versions).")
print()
p90_stats[["null_pct", "mean", "std", "min", "max"]]

Per-90 metrics: 46 columns
  Tag: REVIEW — these are the raw counting stats normalized to 90 min.
  If z-scores are kept, these may be redundant (z-scores = position-adjusted versions).



,null_pct,mean,std,min,max
Aerials per 90,0.000,3.856,2.899,0.000,32.376
Aerials won per 90,0.000,1.912,1.500,0.000,16.832
Assists per 90,0.000,0.059,0.080,0.000,0.783
Attacking aerials won per 90,0.000,0.790,1.045,0.000,13.998
Ball progression (xT) per 90,0.000,0.116,0.066,0.001,0.547
Ball recoveries per 90,0.000,2.450,1.309,0.000,8.411
Ball runs (xT) per 90,0.000,0.021,0.019,0.000,0.194
Box entries per 90,0.000,0.249,0.365,0.000,4.108
Carries (xT) per 90,0.000,0.063,0.032,0.001,0.301
Counterpressing per 90,0.000,0.310,0.236,0.000,1.654


### 3.7 Performance Metrics — Z-Scores (position-adjusted)

In [12]:
z_cols_from = [c for c in fdf.columns if c.startswith("from_z_score_")]
z_cols_to = [c for c in fdf.columns if c.startswith("to_z_score_")]

z_stats = fdf[z_cols_from].describe().T[["count", "mean", "std", "min", "max"]]
z_stats["null_pct"] = (1 - z_stats["count"] / len(fdf)) * 100
z_stats = z_stats.rename(index=lambda x: x.replace("from_z_score_", ""))

print(f"Z-score columns (from_): {len(z_cols_from)}")
print(f"Z-score columns (to_):   {len(z_cols_to)}")
print("  Tag: REVIEW — these are the position-adjusted version of per-90 metrics.")
print("  Most useful representation if the model should compare players across positions fairly.")
print()
z_stats[["null_pct", "mean", "std", "min", "max"]]

Z-score columns (from_): 75
Z-score columns (to_):   75
  Tag: REVIEW — these are the position-adjusted version of per-90 metrics.
  Most useful representation if the model should compare players across positions fairly.



,null_pct,mean,std,min,max
Aerials per 90,0.005,0.048,0.939,-2.603,8.052
Aerials won %,0.025,0.086,0.873,-4.802,3.722
Aerials won per 90,0.005,0.069,0.940,-2.362,6.554
Assists per 90,3.492,0.008,0.876,-1.557,8.570
Attacking aerials won %,9.505,0.058,0.853,-3.706,3.536
Attacking aerials won per 90,4.196,0.056,0.950,-2.135,9.046
Ball progression (xT) per 90,0.005,0.058,0.911,-2.492,6.016
Ball recoveries per 90,0.005,0.068,0.884,-2.680,3.955
Ball runs (xT) per 90,0.743,-0.001,0.897,-2.134,7.487
Box entries per 90,8.502,-0.020,0.849,-2.098,7.357


### 3.8 Metric Layer Redundancy Check

Quick correlation between a quality score, its per-90 equivalent, and its z-score to quantify overlap.

In [13]:
# Pick a few metrics that exist across all three layers
triplets = [
    ("from_Finishing",          "from_Goals per 90",              "from_z_score_Goals per 90"),
    ("from_Passing quality",    "from_Passes (xT) per 90",       "from_z_score_Passes (xT) per 90"),
    ("from_Pressing",           "from_Counterpressing per 90",    "from_z_score_Counterpressing per 90"),
    ("from_Dribbling",          "from_Dribbles (xT) per 90",     "from_z_score_Dribbles (xT) per 90"),
    ("from_Active defence",     "from_Defensive actions per 90",  "from_z_score_Defensive actions per 90"),
]

print("Correlation between metric layers (from_ side):")
print(f"{'Quality Score':<25s} {'vs Per-90':>10s} {'vs Z-Score':>12s} {'Per-90 vs Z':>12s}")
print("-" * 62)
for qs, p90, zs in triplets:
    if all(c in fdf.columns for c in [qs, p90, zs]):
        sub = fdf[[qs, p90, zs]].dropna()
        r_qs_p90 = sub[qs].corr(sub[p90])
        r_qs_z = sub[qs].corr(sub[zs])
        r_p90_z = sub[p90].corr(sub[zs])
        label = qs.replace("from_", "")
        print(f"  {label:<23s} {r_qs_p90:>10.3f} {r_qs_z:>12.3f} {r_p90_z:>12.3f}")

print()
print("Takeaway: If per-90 ↔ z-score correlation is very high (~0.95+),")
print("keeping both is redundant. Choose one layer + quality scores if they add info.")

Correlation between metric layers (from_ side):
Quality Score              vs Per-90   vs Z-Score  Per-90 vs Z
--------------------------------------------------------------
  Finishing                    0.560        0.923        0.613
  Passing quality              0.709        0.920        0.773
  Pressing                     0.534        0.734        0.708
  Dribbling                    0.537        0.862        0.611
  Active defence               0.431        0.886        0.484

Takeaway: If per-90 ↔ z-score correlation is very high (~0.95+),
keeping both is redundant. Choose one layer + quality scores if they add info.


### 3.9 Team Stats Columns

75 team-level aggregate stats per side. These describe the tactical/performance context of the team the player was in.

In [14]:
ts_from = [c for c in fdf.columns if c.startswith("from_team_stats_")]
ts_to = [c for c in fdf.columns if c.startswith("to_team_stats_")]

ts_stats = fdf[ts_from].describe().T[["count", "mean", "std", "min", "max"]]
ts_stats["null_pct"] = (1 - ts_stats["count"] / len(fdf)) * 100
ts_stats["near_zero_var"] = ts_stats["std"] < 0.001
ts_stats = ts_stats.rename(index=lambda x: x.replace("from_team_stats_", ""))

print(f"Team stats columns (from_): {len(ts_from)}")
print(f"Team stats columns (to_):   {len(ts_to)}")
print(f"Columns with any nulls:     {(ts_stats['null_pct'] > 0).sum()}")
print(f"Columns with near-zero var: {ts_stats['near_zero_var'].sum()}")
print()

# Group team stats by category for easier review
categories = {
    "Results & Points": [c for c in ts_from if any(k in c for k in ["goals", "points", "xpts", "win_prob", "goal_diff"])],
    "Shooting": [c for c in ts_from if any(k in c for k in ["shots", "xg", "box_touch", "box_entr"])
                 and "recovery" not in c and "within" not in c],
    "Possession & Build-up": [c for c in ts_from if any(k in c for k in ["possession", "pass_tempo", "field_tilt",
                              "ball_in_play", "long_ball", "buildup", "forward_pass"])],
    "Defensive": [c for c in ts_from if any(k in c for k in ["defensive", "recovery", "ppda", "time_to_def",
                  "counterpressing", "time_to_recovery", "recoveries_within"])],
    "Transition": [c for c in ts_from if any(k in c for k in ["within_10s", "after_recovery"])],
    "Discipline & Set Pieces": [c for c in ts_from if any(k in c for k in ["foul", "card", "corner", "penalt",
                                "offside", "throwin"])],
    "Attack Style": [c for c in ts_from if any(k in c for k in ["sustained", "direct", "crosses_per",
                     "dribbles_per", "shots_per_final", "final_third"])
                     and "recovery" not in c],
}

assigned = set()
for cat, cols in categories.items():
    assigned.update(cols)
categories["Other"] = [c for c in ts_from if c not in assigned]

for cat, cols in categories.items():
    if cols:
        print(f"\n  {cat} ({len(cols)} cols):")
        for c in sorted(cols):
            print(f"    {c.replace('from_team_stats_', '')}")

print(f"\n  Tag: REVIEW — These are all potentially useful as context features.")
print(f"  Consider: do you want team stats as raw values, or as differences (to - from)?")
print(f"  Some may be redundant (e.g., multiple shooting metrics).")

Team stats columns (from_): 74
Team stats columns (to_):   74
Columns with any nulls:     74
Columns with near-zero var: 0


  Results & Points (8 cols):
    goal_difference
    goals
    np_goals
    own_goals
    points
    win_probability_pct
    xpts
    xpts_points_diff

  Shooting (20 cols):
    box_entries
    box_entries_from_carries_pct
    box_entries_from_crosses_pct
    box_touches
    high_opportunity_shots
    np_shots
    np_xg
    np_xg_per_shot
    num_box_entries
    num_shots_from_direct_attacks
    num_shots_from_sustained_attacks
    shots
    shots_from_direct_attacks_pct
    shots_from_outside_box_pct
    shots_from_sustained_attacks_pct
    shots_on_target
    shots_per_final_third_pass
    xg
    xg_from_direct_attacks
    xg_from_sustained_attacks

  Possession & Build-up (16 cols):
    ball_in_play_minutes
    ball_possession_pct
    buildups_from_goalkicks_pct
    crosses_per_final_third_possession
    dribbles_per_final_third_possession
    field_tilt_pct
 

### 3.10 Full Column Summary Table

One consolidated view of every column with its tag for quick decision-making.

In [15]:
def tag_all_columns(columns):
    """Assign a tag and group to every column."""
    results = []

    drop_exact = {
        "wyscout_image_url", "wyscout_first_name", "wyscout_last_name", "short_name",
        "wy_player_id", "birth_date", "transfer_type", "tm_player_id",
        "competition_start_date", "first_played_date", "last_played_date",
        "tm_transfer_date", "tm_contract_until_date",
        "from_competition", "to_competition", "to_position",
        "from_team_id", "to_team_id",
    }
    keep_exact = {"player_season_age", "wyscout_height", "wyscout_weight",
                  "tm_remaining_contract_days", "from_position"}
    review_exact = {"wyscout_foot", "wyscout_role", "wyscout_birth_country", "wyscout_passport",
                    "tm_transfer_value", "tm_transfer_fee",
                    "from_Minutes", "to_Minutes", "from_season", "to_season",
                    "from_matches_pct", "from_minutes_pct", "to_matches_pct", "to_minutes_pct"}

    for col in columns:
        if col in drop_exact:
            results.append({"column": col, "group": "identity/meta", "tag": "DROP"})
        elif col in keep_exact:
            results.append({"column": col, "group": "identity/meta", "tag": "KEEP"})
        elif col in review_exact:
            results.append({"column": col, "group": "identity/meta", "tag": "REVIEW"})
        elif col.startswith(("from_comp_", "to_comp_")):
            base = col.split("comp_")[1]
            if base in ("start_date", "end_date", "completed", "season_id"):
                results.append({"column": col, "group": "competition", "tag": "DROP"})
            else:
                results.append({"column": col, "group": "competition", "tag": "REVIEW"})
        elif col.startswith(("from_z_score_", "to_z_score_")):
            results.append({"column": col, "group": "z_scores", "tag": "REVIEW"})
        elif col.startswith(("from_team_stats_", "to_team_stats_")):
            results.append({"column": col, "group": "team_stats", "tag": "REVIEW"})
        elif col.startswith(("from_", "to_")):
            # Remaining from_/to_ are quality scores and per-90 metrics
            if "per 90" in col:
                results.append({"column": col, "group": "per_90", "tag": "REVIEW"})
            else:
                results.append({"column": col, "group": "quality_scores", "tag": "REVIEW"})
        else:
            results.append({"column": col, "group": "unknown", "tag": "REVIEW"})

    return pd.DataFrame(results)

summary = tag_all_columns(fdf.columns)

print("=== Tag counts ===")
print(summary["tag"].value_counts().to_string())
print()
print("=== Tag counts by group ===")
print(summary.groupby(["group", "tag"]).size().unstack(fill_value=0).to_string())
print()
print(f"Total DROP:   {(summary['tag'] == 'DROP').sum()} columns")
print(f"Total KEEP:   {(summary['tag'] == 'KEEP').sum()} columns")
print(f"Total REVIEW: {(summary['tag'] == 'REVIEW').sum()} columns")

=== Tag counts ===
tag
REVIEW    508
DROP       26
KEEP        5

=== Tag counts by group ===
tag             DROP  KEEP  REVIEW
group                             
competition        8     0       8
identity/meta     18     5      14
per_90             0     0      92
quality_scores     0     0      96
team_stats         0     0     148
z_scores           0     0     150

Total DROP:   26 columns
Total KEEP:   5 columns
Total REVIEW: 508 columns


In [16]:
# Show all DROP columns
print("=== Columns tagged DROP (safe to remove) ===")
drop_cols = summary.loc[summary["tag"] == "DROP", "column"].tolist()
for c in drop_cols:
    print(f"  {c}")
print(f"\n  → {len(drop_cols)} columns to drop")

=== Columns tagged DROP (safe to remove) ===
  wy_player_id
  wyscout_first_name
  wyscout_image_url
  wyscout_last_name
  short_name
  birth_date
  transfer_type
  competition_start_date
  first_played_date
  last_played_date
  tm_player_id
  tm_transfer_date
  tm_contract_until_date
  from_competition
  from_team_id
  from_comp_completed
  from_comp_end_date
  from_comp_season_id
  from_comp_start_date
  to_competition
  to_position
  to_team_id
  to_comp_completed
  to_comp_end_date
  to_comp_season_id
  to_comp_start_date

  → 26 columns to drop


In [17]:
# Show all REVIEW columns grouped
print("=== Columns tagged REVIEW (need your decision) ===\n")
for group in summary["group"].unique():
    group_review = summary.loc[(summary["tag"] == "REVIEW") & (summary["group"] == group), "column"].tolist()
    if group_review:
        print(f"  [{group}] — {len(group_review)} columns:")
        for c in group_review:
            print(f"    {c}")
        print()

=== Columns tagged REVIEW (need your decision) ===

  [identity/meta] — 14 columns:
    wyscout_birth_country
    wyscout_foot
    wyscout_passport
    wyscout_role
    tm_transfer_value
    tm_transfer_fee
    from_Minutes
    from_season
    from_matches_pct
    from_minutes_pct
    to_Minutes
    to_season
    to_matches_pct
    to_minutes_pct

  [quality_scores] — 96 columns:
    from_Active defence
    from_Aerial threat
    from_Aerials won %
    from_Attacking aerials won %
    from_Box threat
    from_Carries (xT) per 100 receptions
    from_Chance prevention
    from_Composure
    from_Defending 1v1 %
    from_Defensive aerials won %
    from_Defensive area (m^2)
    from_Defensive duels won %
    from_Defensive heading
    from_Defensive line height (m)
    from_Dribbles success %
    from_Dribbling
    from_Effectiveness
    from_Finishing
    from_Goals - xG
    from_Goals per box touch
    from_High turnovers per low reception
    from_Hold-up play
    from_Intelligent def

---

## Next Steps

After running this notebook and reviewing the output:

1. **Decide on the DROP columns** — the ones tagged DROP above are safe to remove
2. **Decide on REVIEW columns** — key decisions:
   - **Metric layer:** Keep quality scores, per-90, z-scores, or a combination?
   - **Team stats:** Keep raw values, compute deltas (to - from), or both?
   - **Financial columns:** Are `tm_transfer_fee` / `tm_transfer_value` features or targets?
   - **Categorical columns:** Encode `wyscout_foot`, `from_position`, `from_comp_name`, etc.?
3. **Create notebook 03** with the final column drops, feature engineering, and model-ready dataset